In [1]:
# 1. IMPORTS & INSTALLATIONS

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Install necessary libraries (safe if already installed)
!pip install spacy
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------- ------------------------------ 2.9/12.8 MB 16.8 MB/s eta 0:00:01
     ------------------------------ --------- 9.7/12.8 MB 24.9 MB/s eta 0:00:01
     --------------------------------------- 12.8/12.8 MB 23.3 MB/s eta 0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
# 2. LOAD TRAINING DATASET (NO HEADERS IN YOUR CSV)

columns = ["id", "entity", "sentiment", "tweet"]

df = pd.read_csv("data/twitter_training.csv", names=columns)

print("Shape:", df.shape)
df.head()
df.info()

Shape: (74682, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         74682 non-null  int64 
 1   entity     74682 non-null  object
 2   sentiment  74682 non-null  object
 3   tweet      73996 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [3]:

# 3. FIX MISSING VALUES (AVOID spaCy ERRORS)

df['tweet'] = df['tweet'].fillna("").astype(str)


In [4]:
# 4. LABEL ENCODING (sentiment → numbers)

le = LabelEncoder()
df['sentiment_encoded'] = le.fit_transform(df['sentiment'])

df[['sentiment', 'sentiment_encoded']].head()

,sentiment,sentiment_encoded
0,Positive,3
1,Positive,3
2,Positive,3
3,Positive,3
4,Positive,3


In [ ]:
# 5. PREPROCESSING USING SPACY

import spacy
nlp = spacy.load("en_core_web_sm")

def preprocess(text):
    """Clean text, remove stopwords & punctuation, apply lemmatization."""
    if not isinstance(text, str):
        text = str(text)

    doc = nlp(text)
    filtered_tokens = []
    
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        filtered_tokens.append(token.lemma_)
    
    return " ".join(filtered_tokens)

# Apply preprocessing (this step takes 3–5 minutes)
df['clean_text'] = df['tweet'].apply(preprocess)
df.head()


In [ ]:
# 6. TRAIN–TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['sentiment_encoded'],
    test_size=0.2,
    random_state=42,
    stratify=df['sentiment_encoded']
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

In [ ]:
# 7. MODEL 1: NAIVE BAYES

clf_nb = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb', MultinomialNB())
])

clf_nb.fit(X_train, y_train)
y_pred_nb = clf_nb.predict(X_test)

print("\n===== Naive Bayes Results =====")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

In [ ]:
# 8. MODEL 2: RANDOM FOREST

clf_rf = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('rf', RandomForestClassifier())
])

clf_rf.fit(X_train, y_train)
y_pred_rf = clf_rf.predict(X_test)

print("\n===== Random Forest Results =====")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

In [ ]:
# 9. LOAD VALIDATION DATASET

test_df = pd.read_csv("data/twitter_validation.csv", names=columns)
test_df['tweet'] = test_df['tweet'].fillna("").astype(str)
test_df['clean_text'] = test_df['tweet'].apply(preprocess)

test_df.head()

In [ ]:

# 10. SAMPLE PREDICTION USING RANDOM FOREST

sample_text = test_df['tweet'][10]
print("\nTweet:", sample_text)
print("Original Sentiment:", test_df['sentiment'][10])

processed = [preprocess(sample_text)]
pred = clf_rf.predict(processed)

print("Predicted Sentiment:", le.inverse_transform(pred)[0])

In [ ]:
# CUSTOM SENTIMENT PREDICTION (YOUR OWN INPUT)

your_text = input("Enter a tweet: ")

cleaned = preprocess(your_text)

pred = clf_rf.predict([cleaned])
predicted_label = le.inverse_transform(pred)[0]

print("\nYour Input:", your_text)
print("Predicted Sentiment:", predicted_label)


In [ ]:
import pickle

pickle.dump(clf_rf, open("model.pkl", "wb"))
pickle.dump(le, open("label_encoder.pkl", "wb"))

print("Files saved successfully!")
